# 9 WorkFlow Analista Jr

### 9.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán
<br>El Analista Jr corre sus scripts en la virtual manchine **desktop-jr** que tiene estas características


*   Normal, paga tarifa completa, nunca es apagada por Google
*   reside en el datacenter de Toronto, Canada
*   64 GB de memoria RAM
*   8 vCPU


En Analista Jr **no** puede utilizar Google Colab porque los 12 GB de dichas maquinas virtuales no son suficientes para el tamaño del dataset que está utilizando.



## 9.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")
if( !require("stringr")) install.packages("stringr")

if( !require("dplyr")) install.packages("dplyr")

#### Parametros

In [ ]:
PARAM <- list()
PARAM$experimento <- "9121_ensemble"


#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/"))

### 9.3.1   Preprocesamiento del dataset

In [ ]:
# 1. Definir la lista de experimentos que querés procesar
# Genera: WF9100_multi_semilla, WF9101_multi_semilla, etc.
experimentos <- paste0("WF910", c(0), "_multi_semilla") 
base_path <- "/content/buckets/b1/exp"

dataset_completo <- data.frame()

# 2. Recorrer cada experimento
for (exp in experimentos) {
  exp_dir <- file.path(base_path, exp)
  
  if (!dir.exists(exp_dir)) {
    warning(paste("No existe el directorio para el experimento:", exp_dir))
    next
  }
  
  # Buscar todos los archivos sueltos que coincidan con la sintaxis prediccion_semilla_*.txt
  archivos_prediccion <- list.files(
    path = exp_dir, 
    pattern = "^prediccion_semilla_.*\\.txt$", 
    full.names = TRUE
  )
  
  cat("\n=== Procesando experimento:", exp, "(", length(archivos_prediccion), "archivos de predicción encontrados ) ===\n")
  
  # 3. Recorrer cada archivo de predicción del experimento actual
  for (archivo in archivos_prediccion) {
    dataset <- read.delim(archivo)
    
    # Extraer el número de semilla desde el nombre del archivo
    nombre_archivo <- basename(archivo)
    num_semilla <- sub("^prediccion_semilla_(.*)\\.txt$", "\\1", nombre_archivo)
    
    # Etiquetamos el experimento y la semilla para trazabilidad
    dataset$Exp <- exp
    dataset$Semilla <- num_semilla
    
    dataset_completo <- bind_rows(dataset_completo, dataset)
    cat("  - Leído:", nombre_archivo, "| Registros:", nrow(dataset), "\n")
  }
}

head(dataset_completo)

# 4. Promediar las probabilidades (Ensemble) por cliente a través de TODOS los experimentos y semillas
tb_prediccion <- dataset_completo %>%
  dplyr::group_by(numero_de_cliente) %>%
  dplyr::summarise(prob = mean(prob, na.rm = TRUE))

head(tb_prediccion)

cat("\nTotal clientes consolidados en ensemble final:", nrow(tb_prediccion), "\n")

setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

fwrite(
  tb_prediccion,
  file = "prediccion.txt",
  sep = "\t"
)



#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle

In [ ]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle
tb_prediccion <- fread(paste0("prediccion.txt"))

PARAM$kaggle$competencia <- "utn-2026-virtual-jr"
PARAM$kaggle$cortes <- seq(1800, 3000, by = 100)
nombre_ensemble <- paste0(str_replace(str_replace(experimentos,"_multi_semilla",""),"WF",""),collapse = "_")

PARAM$modelo <- paste0("ensemble_",nombre_ensemble)
  
# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle", showWarnings= FALSE)

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=ensemble_", nombre_ensemble,"_10semillas",
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  Sys.sleep(30)
  cat(salida, "\n")
}

In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

write_yaml( PARAM, file="PARAM.yml")

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")